# Milestone 5 Lab, notebook 7 of 7: The same question, from loaded files

## Where we are

| Notebook | What you do |
|---|---|
| 1. Files arrive, by hand | Load the files the naive way, and watch three things go wrong |
| 2. A loader that is safe to repeat | Fingerprint each file, check its columns, and choose a rule for every file |
| 3. AI reads a changed file | Ask a model what a renamed column means, and score its suggestions |
| 4. A workflow that can wait | Build the loader as a graph that pauses for your approval |
| 5. Crash and carry on | Restart the kernel on purpose, and find the file still waiting |
| 6. A swarm of file workers | Load fifty files side by side, and score the run for quality, cost, and time |
| **7. The same question, from loaded files** (you are here) | Approve the waiting files, run the pipeline, and ask the leader's question |

Model requests in this notebook: **about 6**. A full run of this lab costs about 3 US cents.

## What this notebook is for

The loader is built. Now connect it to everything from milestones 1 to 4.

You load the fifty files, approve the six that wait, and run the pipeline on the records that were loaded. Then you check the one thing that must not have changed: **the answer**. The leader's question should get the same figures as in milestone 4, because only the way the records arrive is different.

> This notebook needs `OPENROUTER_API_KEY` in your `.env` file.

In [ ]:
import pandas as pd
from lab_helpers import (
    LAB,
    MAIN_QUESTION,
    check_key_against_calculated,
    count_outcomes,
    load_result,
    reset_lab,
    show_answer,
    show_doc,
)

reset_lab()

## Step 1: Load the batch

Fifty files, five at a time, with the model suggesting a mapping where one is needed.

In [ ]:
from retail_profit_stack.agent import make_model
from retail_profit_stack.ingest.dispatch import load_batch

results = load_batch(places=LAB, model=make_model(timeout=30), max_concurrency=5)

count_outcomes(results)

## Step 2: Your desk

Six files are waiting for you. For each one you see what is missing, what is unknown, and what the model suggests, with its confidence.

In [ ]:
from retail_profit_stack.ingest.workflow import answer, build_file_workflow, waiting_files

workflow = build_file_workflow(LAB)
questions = waiting_files(workflow, LAB)

rows = []
for question in questions:
    rows.append({
        "file": question["file_name"],
        "missing": question["missing"],
        "suggested mapping": question["suggested_mapping"],
        "confidence": question["suggestion"]["confidence"],
    })

pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows)

Read the note on any file marked low confidence before you approve it. That is what the confidence is for.

In [ ]:
for question in questions:
    if question["suggestion"]["confidence"] == "low":
        print(question["file_name"])
        print("  ", question["suggestion"]["note"])

In real work you would check a low-confidence mapping with whoever sent the file. Here we planted the changes, so we know the suggestions are right if they match the answer key.

In [ ]:
from retail_profit_stack.ingest.mapping import load_mapping_cases

truth = load_mapping_cases()

for question in questions:
    matches = question["suggested_mapping"] == truth[question["file_name"]]
    print(f"{question['file_name']:30} suggestion matches the truth: {matches}")

Approve each file, with the true mapping. Passing the mapping yourself is how a person **corrects** a suggestion: if the model had been wrong, your mapping is the one that is used and saved.

In [ ]:
for question in questions:
    file_name = question["file_name"]
    done = answer(workflow, file_name, approved=True, person="your name", mapping=truth[file_name])
    print(f"{done['outcome']:8} {file_name:30} {done['rows_loaded']} rows")

## Step 3: The load record

Every file that arrived is in the load record, with what happened to it.

In [ ]:
from retail_profit_stack.ingest import ledger

record = pd.DataFrame(ledger.files_seen(LAB.state_file))

record["outcome"].value_counts()

48 loaded, 1 skipped, 1 rejected. Here are the files a finance analyst would ask about: everything that did not simply load by rule 2.

In [ ]:
record[record["rule"] != 2][["file_name", "outcome", "rule", "rows_loaded", "approved_by", "reason"]]

Your approvals were saved, by feed.

In [ ]:
print(LAB.mappings_file.read_text(encoding="utf-8"))

So what happens when the same batch arrives again tomorrow? Nothing is asked, and nothing is loaded.

In [ ]:
again = load_batch(places=LAB, max_concurrency=5)

count_outcomes(again)

## Step 4: The pipeline, on loaded records

The pipeline from milestones 2 to 4 has one change: it builds each source table from the loaded pieces. Run it on what the lab loaded.

In [ ]:
from lab_helpers import LAB_WAREHOUSE

from retail_profit_stack.transform.pipeline import run_pipeline

run = run_pipeline("sql", source_dir=LAB.loaded, warehouse_dir=LAB_WAREHOUSE, note="Loaded from fifty files")

print(run.run_id, "- lineage adds up:", run.record["lineage_adds_up"])
pd.DataFrame(run.record["inputs"]).T

The run record now says how many **files** each table was built from, and its fingerprint is worked out from those files. Load one more file and the fingerprint changes, so milestone 4's run comparison would show it.

The checks ran as before, on the loaded records.

In [ ]:
run.findings[["finding_id", "check_name", "month", "product_id", "outcome", "record_count"]]

The same seven findings as milestone 3. Notice what the loader did **not** do: it did not remove the duplicated records inside the files. Finding duplicated records is the check's job, and it still does it. The loader's job was the duplicated **file**.

## Step 5: The figure you traced by hand

In milestone 4 you traced the trail water bottle's October profit to 754 records. Trace it again, from files this time.

In [ ]:
from retail_profit_stack.lineage.lineage import record_counts

lineage_key = load_result("answer_key_lineage")

profit = run.tables["monthly_product_profit"].to_pandas()
bottle = profit[(profit["product_id"] == "P001") & (profit["month"] == "2026-10")].iloc[0]

print("Contribution profit:", bottle["contribution_profit"], "   answer key:", lineage_key["contribution_profit"])
print("Records behind it:  ", record_counts(run.lineage, "2026-10", "P001"))
print("Answer key:         ", lineage_key["record_counts"])

The same figure, from the same 754 records. Those records reached us in a dozen different files, three of them with renamed columns.

## Step 6: The real folders, and the whole answer key

The lab worked in folders of its own. The production loader uses the real ones, where the three mappings are already approved. `make_history` loads the files there and makes the two runs milestone 4 starts with.

In [ ]:
from retail_profit_stack.make_history import make_history

for made in make_history():
    print(made.run_id, "-", made.record["note"])

In [ ]:
answer_key = load_result("answer_key")

check_key_against_calculated(answer_key)

Every line of the answer key from milestone 3 still matches. Nothing about the answer has moved.

## Step 7: The leader's question

The analyst, its seven tools, and the verifier are exactly as in milestone 4. They do not know or care that the records now come from files.

In [ ]:
from retail_profit_stack.agent import run_analyst

analyst_run = run_analyst(MAIN_QUESTION)

show_answer(analyst_run.answer)

In [ ]:
print("Verified:", analyst_run.verification.passed, "  Run:", analyst_run.run_id)
print([check.name for check in analyst_run.verification.checks if not check.passed])

If a check failed, run step 7 again. Milestone 4 measured how often this question passes, and it is not every time.

## Step 8: Where this milestone leaves us

| | Before this milestone | After |
|---|---|---|
| How records arrive | Seven complete tables, simply there | Fifty files, loaded one piece at a time |
| The same file twice | Would be counted twice | Skipped, by its content's fingerprint |
| A renamed column | Silently empty figures | Noticed; AI suggests, a person approves, code applies |
| A missing column | Loaded anyway | Rejected, and kept in the load record |
| A crash during a load | Start again and hope | Carry on from the last checkpoint; nothing loaded twice |
| Many files | One at a time | Side by side, with a cap and a retry |
| The answer | The figures of milestone 4 | The same figures |

## Step 9: From the lab to production

In [ ]:
show_doc("LLD.md", "12. Where each part is built")

To load the files and run the tests, open a terminal in this milestone's folder and run:

```
uv run python -m retail_profit_stack.make_history
uv run pytest
```

The tests need no key and no network. Where a model is needed they use a scripted fake.

## Your assignment

1. **Let the person say why.** When a mapping is refused, the file is rejected and the reason is only "refused". Change the `approval` node in `retail_profit_stack/ingest/workflow.py` so the answer can carry a `note`, and save the note in the load record's reason. Refuse `shipments_2026-10-26.csv` with a note and read it back.
2. **A new kind of file problem.** Copy `refunds_2026-10-26.csv` in the landing folder to a new name, and in the copy rename `refund_amount` to `amount_refunded`. Load it in the lab's folders. Which rule applies? What does the model suggest? What happens to the copy after you approve it, and why? (Look at its rows.)
3. **Find the limit.** In notebook 6, write a stand-in that fails one time in three, for every file. How many attempts does the `RetryPolicy` need before a batch of fifty always completes?

## What we now know

- The pipeline of milestones 2 to 4 runs unchanged on records built from files.
- Every figure, every finding, and every lineage count equals milestone 4's. Only the way in changed.
- The loader handles the duplicated **file**. The check still handles the duplicated **record**. One job per part.
- Each file's fate is written down, and each approval carries a person's name.
- The same batch can arrive again and nothing happens. That is what "repeatable" means.

You have finished milestone 5. Open `milestone_5_recap.html` in a browser to see what you built.